# 22 · E1i: fit, don't select. Does a slow-current rung beat L1 when both are *fitted*?

**Where Notebooks 19–21 left off.** With every model started from its exact resting state (Notebook 21):
* a global slow K⁺-like current added to the fitted L1, with its two numbers (ρ, τ_n) **selected** on training pairs from a grid, improves cross-validated held-out FEVE by **+0.025** [+0.011, +0.041];
* about half of that a rescaled L1 also gets (+0.0125, not significant); the other half needs the slow dynamics (+0.0125 [+0.004, +0.024] over the rescaled L1);
* the null control was clean;
* E1's 30-s burn-in, although 0.02–0.05 short of rest, barely changes E1's predictions (PROTOCOL_DEPENDENT = False). E1's results and the substrate tests (05, 16–18) stand.

**What has not been tested.** Nothing has been *fitted* so far: the slow current was added to a fixed L1, and "beats the rescaled L1" means beating one direction inside the L1 family, not the best L1. The decisive comparison fits both families with the same optimizer, the same budget and the same start.

| model | start | fitted |
|---|---|---|
| `L1_refit` | the E1 fit | all L1 parameters, `STEPS` more steps |
| `L1s_sel` | the E1 fit + the slow current Notebook 21 selected on this fold's training pairs | all L1 parameters + ρ, τ_n |
| `L1s_off` | the E1 fit + the slow current nearly off (ρ = 0.01, τ_n = 10 s), E1's own nested warm start | the same |
| `L1s_grid` | as `L1s_sel` | nothing (Notebook 21's model, in the ladder's own simulator) |

The new rung `L1s` lives in `ladder.py` and has 2 parameters more than L1. Its gate tracks V during the burn-in, so at stimulus onset it sits at rest: the fix from Notebooks 20–21 is built in.

`L1s_off` answers a second question. **Could E1's nested warm-start procedure have found the effect at all?** E1's L2 started its new mechanism "nearly off" and gained almost nothing.

In [1]:
#@title Setup — run first (works locally on a Mac/PC and on Colab)
import os, sys, subprocess
REPO_URL   = "https://github.com/aslansd/Closure-Bench.git"  # Colab only: clone from here
DRIVE_PATH = "/content/drive/MyDrive/Closure-Bench"          # Colab only: ...or use a Drive copy
IN_COLAB = "google.colab" in sys.modules

def _find_package():
    # 1) explicit location, 2) the repo this notebook lives in, 3) Colab clone / Drive
    cands = [os.environ.get("CLOSUREBENCH_ROOT", ""), ".", "..", "/content/Closure-Bench", DRIVE_PATH]
    for root in cands:
        if root and os.path.isdir(os.path.join(root, "closurebench")):
            return os.path.abspath(root)
    if IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wormneuroatlas", "optax"], check=False)
        if subprocess.run(["git", "clone", "-q", REPO_URL, "/content/Closure-Bench"]).returncode == 0:
            return "/content/Closure-Bench"
        from google.colab import drive
        drive.mount("/content/drive")
        if os.path.isdir(os.path.join(DRIVE_PATH, "closurebench")):
            return DRIVE_PATH
    raise RuntimeError("closurebench not found. Locally: open this notebook from Closure-Bench/notebooks/, "
                       "or set the environment variable CLOSUREBENCH_ROOT to the Closure-Bench folder.")

ROOT = _find_package()
sys.path.insert(0, ROOT)
try:
    import jax, optax, wormneuroatlas  # noqa: F401
except ImportError as e:
    raise ImportError(f"{e}. Locally, create the environment first: see README.md "
                      "('Run locally'), then select the 'Python (closurebench)' kernel.") from None
RESULTS = os.environ.get("CLOSUREBENCH_RESULTS", os.path.join(ROOT, "results"))
os.makedirs(RESULTS, exist_ok=True)
from closurebench import config as CBC
MODE = CBC.get_mode()        # 'smoke' | 'laptop' | 'full'  (env CLOSUREBENCH_MODE, default: full on GPU, laptop on CPU)
print("closurebench:", ROOT); print("results:", RESULTS); print(CBC.describe_machine()); print("MODE:", MODE)

closurebench: /Users/asataryd/Documents/ENI-Projects/Aslan/NeuroAILab/Githubs/Closure-Bench-Mac
results: /Users/asataryd/Documents/ENI-Projects/Aslan/NeuroAILab/Githubs/Closure-Bench-Mac/results
Darwin arm64 | python 3.11.5 | jax 0.10.2 on cpu (cpu) | CPU cores: 8
MODE: laptop


In [2]:
import json, glob, pickle, time, datetime, numpy as np, matplotlib.pyplot as plt, jax
import closurebench
assert tuple(int(x) for x in closurebench.__version__.split(".")[:3]) >= (0, 19, 0), \
    f"Notebook 22 needs closurebench >= 0.19.0 (found {closurebench.__version__}); copy the full package and restart the kernel"
from closurebench import data as D, ladder as lad, metrics as M_, substrates as SB, conductance as CD, intrinsic as IN
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
CONF = {"smoke":  dict(N_COLS=10, FOLDS_USED=1, STEPS=20),
        "laptop": dict(N_COLS=None, FOLDS_USED=None, STEPS=300),
        "full":   dict(N_COLS=None, FOLDS_USED=None, STEPS=1000)}[MODE]
globals().update(CONF)
N_BOOT, E1_THRESHOLD = 2000, 0.05
TAG = "" if MODE == "full" else f"_{MODE}"

# ---- the fitted L1 models of Notebook 04 (as in Notebook 05); smoke falls back to a synthetic L1 if they are absent
E1_TAG = "_laptop" if MODE == "smoke" else TAG
try:
    e1_result = json.load(open(os.path.join(RESULTS, f"E1_result{E1_TAG}.json")))
    e1_plan = json.load(open(os.path.join(RESULTS, e1_result["preregistration"])))
    assert e1_result["L_star"] == "L1", "conductance.py implements L1"
    c = e1_plan["config"]
    ds = D.Dataset.load(os.path.join(RESULTS, "atlas_dataset.npz"))
    if c["n_neurons"] < ds.N:
        mk = ds.mask()[0]; score = mk.sum(1).astype(float); score[ds.stim] += mk.sum(0)
        ds = D.subset(ds, np.sort(np.argsort(-score)[:c["n_neurons"]]))
    st = lad.Structure.from_dataset(ds, tie=c["tie_classes"]); cfg = lad.SimConfig(**c["sim"])
    folds = M_.kfold_pairs(ds, c["k_folds"], c["fold_seed"])
    ck1 = os.path.join(RESULTS, f"e1_checkpoints{E1_TAG}")
    models = [pickle.load(open(os.path.join(ck1, f"fold{f}_L1_final.pkl"), "rb"))["params"] for f in range(c["k_folds"])]
    SOURCE = f"E1 fits ({E1_TAG.strip('_') or 'full'})"
except (FileNotFoundError, KeyError) as err:
    if MODE != "smoke":
        raise
    print("E1 fits not found (", err, ") -> smoke test on a synthetic L1")
    ds, p_true, _ = lad.make_synthetic(lad.random_wiring(N=30, S=10, seed=1, strains=("wt",)), true_level="L1", seed=1)
    st = lad.Structure.from_dataset(ds); cfg = lad.SimConfig(); folds = [None]; models = [p_true]; SOURCE = "synthetic L1"
rng = np.random.default_rng(0)
COLS = np.sort(rng.choice(ds.S, N_COLS, replace=False)) if N_COLS and N_COLS < ds.S else np.arange(ds.S)
FOLDS = list(range(len(models)))[:FOLDS_USED] if FOLDS_USED else list(range(len(models)))
print(f"source: {SOURCE} | {ds.N} neurons | {len(COLS)} stimuli | fold models {FOLDS}")
if folds[0] is None:                       # smoke fallback: a synthetic L1 has no folds -> make two (negative control: no adaptation in the truth)
    folds = M_.kfold_pairs(ds, 2, 0); models = [models[0]] * 2; FOLDS = [0, 1]
LR = float(c.get("lr", 1e-2)) if SOURCE.startswith("E1") else 1e-2
print(f"{STEPS} optimizer steps per fit, learning rate {LR}, same budget for every model")

source: E1 fits (laptop) | 120 neurons | 120 stimuli | fold models [0, 1, 2, 3, 4]
300 optimizer steps per fit, learning rate 0.02, same budget for every model


### Pre-registration (before §1)

Recorded in `results/E1i_preregistration*.json`.
* Fits use each fold's training pairs, E1's loss, regularisation and learning rate, with the same number of steps for every model.
* Held-out predictions (the ladder's simulator, as in E1) are assembled across folds, WT pairs are scored, and 2000 bootstrap draws resample stimulated neurons.

The rules:
* **FIT_KEEPS_GAIN** (primary): `L1s_sel` beats `L1_refit`, CI lower bound > 0. The slow current helps beyond the best L1 the same optimizer finds from the same start.
* **FIT_FINDS_IT:** `L1s_off` beats `L1_refit`, CI lower bound > 0. E1's warm-start procedure can reach the effect.
* **CLOSED_LEVEL_DEEPER:** FIT_KEEPS_GAIN **and** `L1s_sel` beats both `L1_refit` and the E1 fit by ≥ 0.05 (E1's sufficiency margin).
* Reported: the fitted (ρ, τ_n) per fold, and every model's held-out FEVE.

In [3]:
prereg = os.path.join(RESULTS, f"E1i_preregistration{TAG}.json")
plan = {"experiment": "E1i fitted slow-current rung vs fitted L1", "mode": MODE, "registered_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "source": SOURCE, "config": {"folds": FOLDS, "steps": STEPS, "lr": LR, "n_boot": N_BOOT, "strain": "wt"},
        "FIT_KEEPS_GAIN": "L1s_sel beats L1_refit on held-out pairs, 95% CI lower bound > 0",
        "FIT_FINDS_IT": "L1s_off beats L1_refit, CI lower bound > 0",
        "CLOSED_LEVEL_DEEPER": f"FIT_KEEPS_GAIN and L1s_sel - max(L1_refit, E1 fit) >= {E1_THRESHOLD}"}
if os.path.exists(prereg):
    plan = json.load(open(prereg)); print("Existing registration (", plan["registered_at"], ") used unchanged.")
else:
    json.dump(plan, open(prereg, "w"), indent=2); print("Registered ->", prereg)

Registered -> /Users/asataryd/Documents/ENI-Projects/Aslan/NeuroAILab/Githubs/Closure-Bench-Mac/results/E1i_preregistration_laptop.json


## 1 · Fits (checkpointed; the longest part, ~10–20 min per fold on a laptop)

In [4]:
CK = os.path.join(RESULTS, "e1i_ck", MODE); os.makedirs(CK, exist_ok=True)
sel = {}
r21 = os.path.join(RESULTS, f"E1h_result{TAG}.json")
if os.path.exists(r21):
    for f, (a, b) in json.load(open(r21))["choices"].items():
        sel[int(f)] = (float(b.split("_")[1]), float(b.split("_")[2])) if b.startswith("B_") else (0.05, 1e3)
print("slow currents selected in Notebook 21 (rho, tau_n):", sel if sel else "not found -> (0.1, 20 s) for every fold")
def split(f):
    test = folds[f]; train = np.zeros_like(test)
    for g in range(len(folds)):
        if g != f: train |= folds[g]
    return train, test
def l1s_init(p, rho, tau_n):
    q = lad.extend_params(p, "L1s", st, jax.random.PRNGKey(0))
    q["rho_s"] = jnp.array(float(np.log(rho / (0.9 - rho)))); q["tau_ns"] = jnp.array(lad.inv_sp(max(tau_n - lad.TAU_MIN, 0.05)))
    return q
import jax.numpy as jnp
FITS = {}
for f in FOLDS:
    train, _ = split(f)
    p0 = jax.tree_util.tree_map(jnp.asarray, models[f])
    rho, tn = sel.get(f, (0.1, 20.0))
    plans = {"L1_refit": ("L1", dict(p0)), "L1s_sel": ("L1s", l1s_init(p0, rho, tn)), "L1s_off": ("L1s", l1s_init(p0, 0.01, 10.0))}
    FITS[f] = {"E1": ("L1", p0), "L1s_grid": ("L1s", l1s_init(p0, rho, tn))}
    for name, (lev, init) in plans.items():
        ck = os.path.join(CK, f"fold{f}_{name}.pkl")
        if os.path.exists(ck):
            FITS[f][name] = (lev, pickle.load(open(ck, "rb"))); continue
        t0 = time.time()
        pf, losses = lad.fit(lev, ds, train_mask=train & ds.mask(), init=init, steps=STEPS, lr=LR, verbose=0, st=st, cfg=cfg)
        pf = jax.tree_util.tree_map(np.asarray, pf)
        pickle.dump(pf, open(ck, "wb")); FITS[f][name] = (lev, pf)
        extra = f" | rho {0.9 / (1 + np.exp(-pf['rho_s'])):.3f}, tau_n {lad.TAU_MIN + np.log1p(np.exp(pf['tau_ns'])):.1f} s" if lev == "L1s" else ""
        print(f"fold {f} {name:9s}: training loss {losses[0]:.4f} -> {losses[-1]:.4f}{extra}  ({time.time() - t0:.0f} s)", flush=True)

slow currents selected in Notebook 21 (rho, tau_n): {0: (0.1, 20.0), 1: (0.1, 10.0), 2: (0.1, 20.0), 3: (0.2, 5.0), 4: (0.1, 20.0)}
fold 0 L1_refit : training loss 0.8453 -> 0.8385  (99 s)
fold 0 L1s_sel  : training loss 0.8903 -> 0.8358 | rho 0.146, tau_n 20.0 s  (97 s)
fold 0 L1s_off  : training loss 0.8451 -> 0.8384 | rho 0.011, tau_n 10.0 s  (95 s)
fold 1 L1_refit : training loss 0.8375 -> 0.8304  (87 s)
fold 1 L1s_sel  : training loss 0.8553 -> 0.8268 | rho 0.149, tau_n 10.0 s  (84 s)
fold 1 L1s_off  : training loss 0.8373 -> 0.8297 | rho 0.012, tau_n 10.0 s  (91 s)
fold 2 L1_refit : training loss 0.8305 -> 0.8246  (98 s)
fold 2 L1s_sel  : training loss 0.8656 -> 0.8171 | rho 0.174, tau_n 20.0 s  (86 s)
fold 2 L1s_off  : training loss 0.8302 -> 0.8219 | rho 0.012, tau_n 10.0 s  (86 s)
fold 3 L1_refit : training loss 0.8350 -> 0.8274  (87 s)
fold 3 L1s_sel  : training loss 0.8552 -> 0.8253 | rho 0.174, tau_n 5.1 s  (85 s)
fold 3 L1s_off  : training loss 0.8348 -> 0.8273 | rho 0.011

## 2 · Held-out comparison

In [5]:
NAMES = ("E1", "L1_refit", "L1s_grid", "L1s_sel", "L1s_off")
cv = {k: np.zeros(ds.mean.shape[:3]) for k in NAMES}
for f in FOLDS:
    _, test = split(f)
    for k in NAMES:
        lev, pp = FITS[f][k]
        R = np.asarray(lad.predict(lev, jax.tree_util.tree_map(jnp.asarray, pp), st, ds.stim, ds.strains, cfg, np.arange(ds.S)))
        cv[k][test] = R[test]
point, boots = M_.bootstrap_feve(cv, ds, list(NAMES), n_boot=N_BOOT, seed=1, strain="wt")
print("held-out FEVE (WT): " + " | ".join(f"{k} {point[k]:.4f}" for k in NAMES))
def diff(a, b):
    d = boots[a] - boots[b]; return point[a] - point[b], np.percentile(d, [2.5, 97.5])
TESTS = {}
for a, b in (("L1_refit", "E1"), ("L1s_grid", "E1"), ("L1s_sel", "L1_refit"), ("L1s_off", "L1_refit"), ("L1s_sel", "E1"), ("L1s_sel", "L1s_grid")):
    g, ci = diff(a, b); TESTS[f"{a}-{b}"] = (g, ci)
    print(f"  {a:>8s} - {b:<8s}: {g:+.4f}  95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}]")
for f in FOLDS:
    for k in ("L1s_sel", "L1s_off"):
        pp = FITS[f][k][1]
        print(f"fold {f} {k}: rho {0.9 / (1 + np.exp(-np.asarray(pp['rho_s']))):.3f}, tau_n {lad.TAU_MIN + np.log1p(np.exp(np.asarray(pp['tau_ns']))):.1f} s")
FIT_KEEPS_GAIN = bool(TESTS["L1s_sel-L1_refit"][1][0] > 0)
FIT_FINDS_IT = bool(TESTS["L1s_off-L1_refit"][1][0] > 0)
CLOSED_LEVEL_DEEPER = bool(FIT_KEEPS_GAIN and point["L1s_sel"] - max(point["L1_refit"], point["E1"]) >= E1_THRESHOLD)
print(f"\nRegistered rules -> FIT_KEEPS_GAIN: {FIT_KEEPS_GAIN} | FIT_FINDS_IT: {FIT_FINDS_IT} | CLOSED_LEVEL_DEEPER: {CLOSED_LEVEL_DEEPER}")

held-out FEVE (WT): E1 0.2683 | L1_refit 0.2699 | L1s_grid 0.2978 | L1s_sel 0.2943 | L1s_off 0.2777
  L1_refit - E1      : +0.0016  95% CI [-0.0102, +0.0116]
  L1s_grid - E1      : +0.0295  95% CI [+0.0156, +0.0454]
   L1s_sel - L1_refit: +0.0244  95% CI [+0.0141, +0.0385]
   L1s_off - L1_refit: +0.0079  95% CI [+0.0027, +0.0157]
   L1s_sel - E1      : +0.0260  95% CI [+0.0148, +0.0407]
   L1s_sel - L1s_grid: -0.0035  95% CI [-0.0218, +0.0145]
fold 0 L1s_sel: rho 0.146, tau_n 20.0 s
fold 0 L1s_off: rho 0.011, tau_n 10.0 s
fold 1 L1s_sel: rho 0.149, tau_n 10.0 s
fold 1 L1s_off: rho 0.012, tau_n 10.0 s
fold 2 L1s_sel: rho 0.174, tau_n 20.0 s
fold 2 L1s_off: rho 0.012, tau_n 10.0 s
fold 3 L1s_sel: rho 0.174, tau_n 5.1 s
fold 3 L1s_off: rho 0.011, tau_n 10.0 s
fold 4 L1s_sel: rho 0.162, tau_n 20.0 s
fold 4 L1s_off: rho 0.011, tau_n 10.0 s

Registered rules -> FIT_KEEPS_GAIN: True | FIT_FINDS_IT: True | CLOSED_LEVEL_DEEPER: False


In [6]:
out = {"source": SOURCE, "steps": STEPS, "heldout": {k: float(point[k]) for k in NAMES},
       "tests": {k: {"point": g, "ci": list(ci)} for k, (g, ci) in TESTS.items()},
       "fitted": {str(f): {k: {"rho": float(0.9 / (1 + np.exp(-np.asarray(FITS[f][k][1]["rho_s"])))),
                               "tau_n": float(lad.TAU_MIN + np.log1p(np.exp(np.asarray(FITS[f][k][1]["tau_ns"]))))} for k in ("L1s_sel", "L1s_off")} for f in FOLDS},
       "FIT_KEEPS_GAIN": FIT_KEEPS_GAIN, "FIT_FINDS_IT": FIT_FINDS_IT, "CLOSED_LEVEL_DEEPER": CLOSED_LEVEL_DEEPER}
json.dump(out, open(os.path.join(RESULTS, f"E1i_result{TAG}.json"), "w"), indent=2, default=float)
print("saved", f"E1i_result{TAG}.json")

saved E1i_result_laptop.json


## Results (laptop mode, executed on the M1: 5 folds × 3 fits of 300 steps, learning rate 0.02)

**Held-out FEVE (WT, cross-validated, ladder simulator as in E1):**

| model | held-out FEVE |
|---|---|
| E1 fit | 0.2683 |
| `L1_refit` | 0.2699 |
| `L1s_grid` (Notebook 21's selection, not fitted) | **0.2978** |
| `L1s_sel` (fitted from the selected slow current) | **0.2943** |
| `L1s_off` (fitted from "nearly off") | 0.2777 |

| comparison | gain | 95 % CI | rule |
|---|---|---|---|
| `L1s_sel` − `L1_refit` | **+0.024** | [+0.014, +0.039] | **FIT_KEEPS_GAIN: yes** |
| `L1s_off` − `L1_refit` | +0.008 | [+0.003, +0.016] | FIT_FINDS_IT: yes (but see below) |
| `L1_refit` − E1 | +0.002 | [−0.010, +0.012] | — |
| `L1s_grid` − E1 | +0.030 | [+0.016, +0.045] | — |
| `L1s_sel` − `L1s_grid` | −0.004 | [−0.022, +0.015] | — |
| CLOSED_LEVEL_DEEPER | | | **no** (gain < 0.05) |

**What the fits did.**
* **ρ grows when it starts near the selected value:** from 0.1–0.2 to **0.15–0.17 on every fold**, a consistent share of the leak.
* **From "nearly off" it hardly moves** (0.010 → 0.011–0.012). The small `L1s_off` gain comes from that sliver plus re-adjusted L1 parameters. The registered rule counts it as "finds it", but in practice **E1's nested warm start cannot grow the mechanism** within E1's budget. That is a concrete reason why E1's adaptation rung L2, warm-started the same way, gained almost nothing.
* **τ_n never moved** from its starting value (20, 10, 5 s) on any fold. The timescale is not identifiable from window-averaged amplitudes, as Notebooks 06–07 predicted. What the data constrain is the *existence and share* of a slow component, not its time constant.
* **More training of L1 itself neither helps nor hurts** (+0.002). Notebook 04's observation that further training lowered held-out FEVE does not reproduce here.
* **Fitting adds nothing over the grid selection** (−0.004, n.s.).

**Outcome: between the first two rows of the table below.** A slow current over seconds is part of the best description of the atlas. It survives selection, a consistent start and fitting, at +0.024–0.030 held-out FEVE (about a tenth of the explained variance). But E1's procedure could not have found it, and it is half of E1's 0.05 sufficiency margin.

**Notebook 23 (E1j)** puts `L1s` back into E1's actual decision: all rungs (including the wiring-only L0, which tied with L1) and all three registered rules.

> **Follow-up (Notebook 23).** On E1's full ladder and rules, `L1s` is the best rung (0.323, both strains) but does not reliably beat the wiring-only L0 (+0.028 [−0.043, +0.103]). E1's decisions are unchanged (L\* = L1, L_needed = L0, v4 = L0).
>
> **Follow-up (Notebook 24).** "More training of L1 itself neither helps nor hurts" does **not** hold for L0. The same 300 refinement steps raise the wiring-only L0 from 0.295 to **0.375**, well above `L1s_sel`, and a slow current on L0 does not beat that (−0.012). This notebook's comparison (L1s vs a refined L1) was fair; E1's comparison of L1 with L0 was not. Notebook 25 refines every rung equally.
>
> **Follow-up (Notebook 25).** A second refinement round confirms FIT_KEEPS_GAIN: `L1s` − L1 = +0.022 [+0.010, +0.040]. But the whole nonlinear family stays about 0.1 below the refined linear L0 (0.421), so the slow current is a better L1, not a deeper closed level.

> **Follow-up (Notebook 29).** Converged, `L1s` − L1 = +0.041 [+0.027, +0.059], almost twice this notebook's +0.024. The whole nonlinear family is no longer below L0: L1s (0.488) is the best rung on the ladder and E1's closed level.

## 3 · Reading the result

| FIT_KEEPS_GAIN | FIT_FINDS_IT | reading |
|---|---|---|
| **yes ← observed** | **yes (registered) ← observed**, though in practice ρ barely grows from "off" | A slow current over seconds is part of the best description of the atlas, and E1's procedure can find it. E1 should be re-run with `L1s` on the ladder. If the gain also reaches 0.05, the closed level is deeper than L1. |
| yes | no | The slow current helps, but **E1's nested warm start cannot reach it**: from "nearly off", the optimizer does not grow it. That is a concrete reason why E1's L2 found nothing. The ladder's warm starts should include a grid over the new mechanism's slow timescales. |
| no | – | Once L1 is properly re-fitted, the slow current adds nothing: Notebooks 19–21's gains reflected an L1 short of its family's optimum. E1's bracket stands, with a better-fitted L1. |

`L1_refit − E1` also tests Notebook 04's observation that more training lowers held-out FEVE (observed here: +0.002, n.s.).

**Limits.**
* A single global slow current (2 numbers). Per-class adaptation is E1's L2, which is less identifiable (Notebook 07).
* `STEPS` steps from the E1 fit is a refinement, not a refit from scratch.